# 03 - Exploratory Data Analysis (EDA)

Notebook ini mengeksplorasi dataset IMDb yang sudah dibersihkan untuk menemukan pola, hubungan antar variabel, dan insight awal.

**Pertanyaan yang ingin dijawab:**

1. Bagaimana sebaran rating dan durasi film?
2. Genre apa yang paling banyak diproduksi dan genre mana yang paling diminati?
3. Bagaimana tren jumlah film dan rating dari tahun ke tahun?
4. Sutradara dan aktor siapa yang paling sering muncul?
5. Variabel apa yang paling berhubungan dengan rating dan pendapatan film?

> Catatan: seluruh analisis menggunakan dataset bersih hasil notebook 02.

## 1. Import Library

Mengimpor `pandas` untuk analisis data tabular.

In [ ]:
# Mengimpor library pandas untuk analisis data
import pandas as pd

## 2. Load Dataset Bersih

Membaca dataset hasil cleaning (`imdb_movie_dataset_clean.csv`).

In [ ]:
# Membaca dataset hasil cleaning
df = pd.read_csv("../data/imdb_movie_dataset_clean.csv")

# Menampilkan ukuran dan 5 baris pertama data
print("Ukuran data:", df.shape)
df.head()

## 3. Statistik Deskriptif

Ringkasan statistik kolom numerik untuk melihat pusat dan sebaran data.

In [ ]:
# Menampilkan statistik deskriptif kolom numerik
df.describe().round(2)

## 4. Distribusi Rating

Film dikelompokkan ke dalam rentang rating untuk melihat pada rentang mana sebagian besar film berada.

In [ ]:
# Mengelompokkan film ke dalam rentang rating
bins = [0, 4, 6, 7, 8, 10]
label = ["< 4", "4 - 6", "6 - 7", "7 - 8", "8 - 10"]

distribusi_rating = (
    pd.cut(df["Rating"], bins=bins, labels=label)
    .value_counts()
    .sort_index()
    .reset_index()
)
distribusi_rating.columns = ["Rentang Rating", "Jumlah Film"]
distribusi_rating

## 5. Distribusi Durasi Film

Mengelompokkan film berdasarkan durasi untuk melihat durasi yang paling umum.

In [ ]:
# Mengelompokkan film berdasarkan durasi (menit)
bins = [0, 90, 110, 130, 200]
label = ["< 90 menit", "90 - 110 menit", "110 - 130 menit", "> 130 menit"]

distribusi_runtime = (
    pd.cut(df["Runtime (Minutes)"], bins=bins, labels=label)
    .value_counts()
    .sort_index()
    .reset_index()
)
distribusi_runtime.columns = ["Rentang Durasi", "Jumlah Film"]
distribusi_runtime

## 6. Genre Paling Banyak Diproduksi

Kolom `Genre` berisi kombinasi 1-3 genre dalam satu sel, sehingga dipecah (*explode*) lebih dulu agar setiap genre dapat dihitung terpisah.

In [ ]:
# Memecah kolom Genre menjadi satu genre per baris
genre_df = df.assign(Genre=df["Genre"].str.split(",")).explode("Genre")
genre_df["Genre"] = genre_df["Genre"].str.strip()

# Menghitung jumlah film untuk setiap genre
jumlah_genre = genre_df["Genre"].value_counts().reset_index()
jumlah_genre.columns = ["Genre", "Jumlah Film"]
jumlah_genre.head(15)

## 7. Rata-rata Rating dan Revenue per Genre

Membandingkan performa tiap genre: seberapa banyak diproduksi, seberapa tinggi ratingnya, dan seberapa besar pendapatannya.

In [ ]:
# Menghitung jumlah film, rata-rata rating, dan rata-rata revenue per genre
ringkasan_genre = (
    genre_df.groupby("Genre")
    .agg(
        jumlah_film=("Title", "count"),
        rata_rating=("Rating", "mean"),
        rata_revenue=("Revenue (Millions)", "mean"),
    )
    .round(2)
    .sort_values("jumlah_film", ascending=False)
    .head(10)
)
ringkasan_genre

## 8. Kombinasi Genre Terbanyak

Selain genre tunggal, menarik untuk melihat kombinasi genre (isi sel asli) yang paling sering muncul.

In [ ]:
# Menghitung kombinasi genre yang paling banyak muncul
top_kombinasi_genre = df["Genre"].value_counts().head(10).reset_index()
top_kombinasi_genre.columns = ["Kombinasi Genre", "Jumlah Film"]
top_kombinasi_genre

## 9. Jumlah Film per Tahun

Melihat banyaknya film yang dirilis setiap tahun pada dataset ini.

In [ ]:
# Menghitung jumlah film per tahun
film_per_tahun = df["Year"].value_counts().sort_index().reset_index()
film_per_tahun.columns = ["Tahun", "Jumlah Film"]
film_per_tahun

## 10. Tren Rating dan Revenue per Tahun

Melihat apakah kualitas dan pendapatan film berubah dari tahun ke tahun.

In [ ]:
# Menghitung rata-rata rating dan revenue per tahun
tren_tahunan = (
    df.groupby("Year")
    .agg(
        jumlah_film=("Title", "count"),
        rata_rating=("Rating", "mean"),
        rata_revenue=("Revenue (Millions)", "mean"),
    )
    .round(2)
)
tren_tahunan

## 11. Sutradara dengan Film Terbanyak

Sutradara yang paling sering muncul pada dataset (bukan berarti film terbaik).

In [ ]:
# Menghitung 10 sutradara dengan jumlah film terbanyak
top_director = df["Director"].value_counts().head(10).reset_index()
top_director.columns = ["Sutradara", "Jumlah Film"]
top_director

## 12. Aktor dengan Film Terbanyak

Kolom `Actors` memuat beberapa nama dalam satu sel, sehingga dipecah lebih dulu.

In [ ]:
# Memecah kolom Actors menjadi satu aktor per baris
aktor_df = df.assign(Actors=df["Actors"].str.split(",")).explode("Actors")
aktor_df["Actors"] = aktor_df["Actors"].str.strip()

# Menghitung 10 aktor dengan jumlah film terbanyak
top_aktor = aktor_df["Actors"].value_counts().head(10).reset_index()
top_aktor.columns = ["Aktor", "Jumlah Film"]
top_aktor

## 13. Korelasi Antar Variabel Numerik

Mengukur kekuatan hubungan antar variabel numerik dengan koefisien korelasi (-1 sampai 1).

In [ ]:
# Menghitung matriks korelasi variabel numerik
kolom_numerik = ["Runtime (Minutes)", "Rating", "Votes", "Revenue (Millions)", "Metascore"]

df[kolom_numerik].corr().round(2)

## 14. Hubungan Popularitas (Votes) dengan Rating dan Revenue

Film dikelompokkan berdasarkan jumlah votes untuk melihat pola rating dan pendapatan pada tiap tingkat popularitas.

In [ ]:
# Mengelompokkan film berdasarkan jumlah votes
bins = [0, 50000, 100000, 250000, 500000, 2000000]
label = ["< 50rb", "50rb - 100rb", "100rb - 250rb", "250rb - 500rb", "> 500rb"]

analisis_votes = (
    df.assign(Kelompok_Votes=pd.cut(df["Votes"], bins=bins, labels=label))
    .groupby("Kelompok_Votes", observed=True)
    .agg(
        jumlah_film=("Title", "count"),
        rata_rating=("Rating", "mean"),
        rata_revenue=("Revenue (Millions)", "mean"),
    )
    .round(2)
)
analisis_votes

## 15. Kesimpulan EDA

**Distribusi data**

- Rating film terkonsentrasi pada rentang **6-8** (731 dari 1000 film: 391 film di rentang 6-7 dan 340 film di rentang 7-8).
- Durasi paling umum adalah **90-110 menit** (418 film), diikuti 110-130 menit (338 film).
- Pendapatan film sangat timpang: median 48,02 juta, sedangkan rata-rata mencapai 79,63 juta.

**Genre**

- Genre terbanyak: **Drama (513 film)**, Action (303), dan Comedy (279).
- Rata-rata rating tertinggi di antara genre besar dipegang **Biography (7,29)** dan **Drama (6,95)**, sedangkan **Horror** paling rendah (6,09).
- Rata-rata pendapatan tertinggi ditemukan pada **Animation (165,03 juta)**, Adventure (130,86), dan Family (117,78).

**Tren tahunan**

- Jumlah film dalam dataset meningkat tajam di tahun terakhir: 2015 (127 film) dan 2016 (297 film).
- Namun rata-rata rating justru menurun: 7,12 (2006) menjadi 6,44 (2016), demikian pula rata-rata pendapatan turun dari 80,95 menjadi 48,93 juta.

**Tokoh**

- Sutradara paling sering muncul: **Ridley Scott (8 film)**, disusul David Yates, M. Night Shyamalan, Paul W.S. Anderson, dan Michael Bay (masing-masing 6 film).
- Aktor paling sering muncul: **Mark Wahlberg (15 film)**, Hugh Jackman (14), Christian Bale dan Brad Pitt (masing-masing 13 film).

**Hubungan antar variabel**

- Korelasi terkuat: **Votes vs Revenue (0,64)** dan **Rating vs Metascore (0,60)**.
- **Votes vs Rating (0,51)** juga cukup kuat, artinya film dengan rating tinggi cenderung mendapat banyak votes.
- Sebaliknya, **Rating vs Revenue hanya 0,21** - rating tinggi tidak otomatis berarti pendapatan besar.
- Film dengan votes di atas 500 ribu memiliki rata-rata rating dan pendapatan paling tinggi dibanding kelompok lainnya.

**Langkah selanjutnya**: hasil temuan ini divisualisasikan pada notebook `04_data_visualization.ipynb`.